# Vežba 5: kNN i logistička regresija

Klasifikacija zasnovana na instancama i na modelu: sigmoidna funkcija, funkcija gubitka, logičke funkcije i granica odlučivanja

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba5.html](https://isatvss.surge.sh/vezba5.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Logistička regresija

### Zadatak 1. Sigmoidna funkcija

Napisati funkciju `sigmoid`, izračunati je za nekoliko vrednosti z i nacrtati je.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


for z in [-10, -2, 0, 2, 10]:
    print(f"g({z:3d}) = {sigmoid(z):.5f}")

z = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(z, sigmoid(z), color="#5b3cc4", linewidth=2.5)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Sigmoidna funkcija g(z) = 1 / (1 + e^(-z))")
plt.xlabel("z")
plt.ylabel("g(z)")
plt.show()

### Zadatak 2. Neuron kao logička funkcija

Za ulaze 0 i 1 i parametre θ = (−10, 20, 20) i θ = (−30, 20, 20) izračunati z, h(x) i izlaz sa pragom 0,5. Ovo je 6. zadatak sa teorijskog testa.

Na ispitu se odgovor obrazlaže tabelom kao u ispisu: za svaki od četiri ulaza izračuna se z, pa iz znaka izlaz. Sa parametrima sa ispita, θ0 = −10 i θ1 = θ2 = 20, hipoteza predstavlja funkciju OR.

In [ ]:
ulazi = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

for naziv, theta in [
    ("theta = (-10, 20, 20)", (-10, 20, 20)),
    ("theta = (-30, 20, 20)", (-30, 20, 20)),
]:
    z = theta[0] + theta[1] * ulazi[:, 0] + theta[2] * ulazi[:, 1]
    tabela = pd.DataFrame({"x1": ulazi[:, 0], "x2": ulazi[:, 1], "z": z})
    tabela["h(x)"] = sigmoid(z).round(5)
    tabela["izlaz"] = (tabela["h(x)"] >= 0.5).astype(int)
    print(naziv)
    print(tabela.to_string(index=False))
    print()

### Zadatak 3. Zašto XOR nije moguć

Isprobati sve kombinacije parametara od −30 do 30 sa korakom 5 i izbrojati one koje daju funkciju XOR.

Nijedna kombinacija ne daje XOR, i to ne zbog izabranog opsega. Jedan neuron deli ravan jednom pravom, a tačke (0, 1) i (1, 0) sa izlazom 1 i tačke (0, 0) i (1, 1) sa izlazom 0 ne mogu se razdvojiti jednom pravom. Za XOR je potreban skriveni sloj, koji se obrađuje na vežbi 9.

In [ ]:
xor = np.array([0, 1, 1, 0])
vrednosti = np.arange(-30, 31, 5)
pogodaka = 0
for t0 in vrednosti:
    for t1 in vrednosti:
        for t2 in vrednosti:
            izlaz = (t0 + t1 * ulazi[:, 0] + t2 * ulazi[:, 1] >= 0).astype(int)
            pogodaka += np.array_equal(izlaz, xor)
print("Isprobanih kombinacija theta:", len(vrednosti) ** 3)
print("Kombinacija koje daju XOR:", pogodaka)

## Funkcija gubitka i granica odlučivanja

### Zadatak 4. Log-loss ručno

Za četiri primera sa poznatom klasom i procenjenom verovatnoćom izračunati gubitak svakog primera i funkciju cene, pa rezultat proveriti funkcijom `log_loss`.

In [ ]:
from sklearn.metrics import log_loss

stvarno = np.array([1, 1, 0, 0])
verovatnoce = np.array([0.9, 0.6, 0.2, 0.95])

gubici = -(stvarno * np.log(verovatnoce) + (1 - stvarno) * np.log(1 - verovatnoce))
print(
    pd.DataFrame(
        {"y": stvarno, "h(x)": verovatnoce, "gubitak": gubici.round(3)}
    ).to_string(index=False)
)
print(f"\nJ = prosek gubitaka = {gubici.mean():.4f}")
print(f"log_loss iz scikit-learn = {log_loss(stvarno, verovatnoce):.4f}")

### Zadatak 5. Granica odlučivanja na podacima o vazduhu

Procenjivati da li će dan imati PM10 iznad 50 µg/m³ na osnovu temperature i najveće brzine vetra. Istrenirati logističku regresiju na danima do kraja 2022, uporediti tačnost sa osnovnim modelom i nacrtati verovatnoću i granicu odlučivanja.

Prekoračenja su češća pri niskoj temperaturi i slabom vetru, zimi kada se greje i kada se vazduh ne meša. Model ima tačnost 0,830 naspram 0,792 osnovnog modela koji uvek procenjuje da prekoračenja nema. Razlika je mala jer se dve klase, kako grafik pokazuje, jako preklapaju. Na vežbi 6 biće jasno zašto je tačnost ovde slaba mera.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

vazduh = pd.read_csv(
    "https://isatvss.surge.sh/podaci/vazduh_nis.csv", parse_dates=["datum"]
)
vazduh = vazduh.dropna(subset=["pm10"])
vazduh["prekoracenje"] = (vazduh["pm10"] > 50).astype(int)
obelezja_2d = ["temperatura", "vetar_max"]
tr = vazduh[vazduh["datum"].dt.year <= 2022]
te = vazduh[vazduh["datum"].dt.year >= 2023]
print(f"Udeo dana sa prekoračenjem u treningu: {tr['prekoracenje'].mean():.3f}")

logisticka = make_pipeline(StandardScaler(), LogisticRegression())
logisticka.fit(tr[obelezja_2d], tr["prekoracenje"])
tacnost_osnovni = 1 - te["prekoracenje"].mean()
tacnost_log = logisticka.score(te[obelezja_2d], te["prekoracenje"])
print(f"Tačnost osnovnog modela na testu: {tacnost_osnovni:.3f}")
print(f"Tačnost logističke regresije na testu: {tacnost_log:.3f}")

mreza_t, mreza_v = np.meshgrid(np.linspace(-8, 32, 300), np.linspace(2, 36, 300))
mreza = pd.DataFrame({"temperatura": mreza_t.ravel(), "vetar_max": mreza_v.ravel()})
p = logisticka.predict_proba(mreza)[:, 1].reshape(mreza_t.shape)

plt.figure(figsize=(7, 5))
plt.contourf(
    mreza_t, mreza_v, p, levels=np.linspace(0, 1, 11), cmap="Oranges", alpha=0.55
)
plt.colorbar(label="h(x): verovatnoća prekoračenja")
plt.contour(mreza_t, mreza_v, p, levels=[0.5], colors="black", linewidths=2)
boje = tr["prekoracenje"].map({0: "#5b3cc4", 1: "#d9480f"})
plt.scatter(tr["temperatura"], tr["vetar_max"], c=boje, s=8, alpha=0.6)
plt.xlabel("Srednja dnevna temperatura (°C)")
plt.ylabel("Najveća brzina vetra (km/h)")
plt.title("Logistička regresija: granica odlučivanja h(x) = 0,5")
plt.show()

## kNN klasifikacija

### Zadatak 6. Uticaj broja suseda

Na istom problemu sa vazduhom istrenirati kNN sa k = 1, 15 i 151, nacrtati oblasti koje model dodeljuje svakoj klasi i uporediti tačnost na treningu i testu.

Sa k = 1 tačnost na treningu je 0,996, a na testu 0,787, ispod osnovnog modela. Sa k = 15 i k = 151 tačnost na testu je 0,829, gotovo ista kao kod logističke regresije. Mali k znači overfitting, a veliki k underfitting, isto kao stepen polinoma na vežbi 4.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

fig, ose = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for osa, k in zip(ose, [1, 15, 151]):
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(tr[obelezja_2d], tr["prekoracenje"])
    tacnost_tr = knn.score(tr[obelezja_2d], tr["prekoracenje"])
    tacnost_te = knn.score(te[obelezja_2d], te["prekoracenje"])
    print(f"k = {k:3d}: tačnost trening = {tacnost_tr:.3f}, test = {tacnost_te:.3f}")
    klasa = knn.predict(mreza).reshape(mreza_t.shape)
    osa.contourf(
        mreza_t, mreza_v, klasa, levels=[-0.5, 0.5, 1.5], colors=["#efe9fb", "#ffe0c7"]
    )
    osa.scatter(tr["temperatura"], tr["vetar_max"], c=boje, s=6, alpha=0.6)
    osa.set_title(f"kNN, k = {k}")
    osa.set_xlabel("Temperatura (°C)")
ose[0].set_ylabel("Vetar (km/h)")
plt.tight_layout()
plt.show()

### Zadatak 7. Nezgode: logistička regresija i kNN

Na skupu nezgoda, sa pripremom sa vežbe 2, uporediti logističku regresiju i kNN sa k = 5 i k = 51.

Logistička regresija ima tačnost 0,747, a kNN 0,721 sa k = 5 i 0,733 sa k = 51. Za procenu 6.638 test nezgoda kNN računa 176.232.262 rastojanja. Na ovom skupu model zasnovan na instancama je i slabiji i sporiji.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode = nezgode.drop_duplicates(subset="id_nezgode").reset_index(drop=True)
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)
vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")
nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month

numericke = ["sat", "dan_u_nedelji", "mesec", "lon", "lat"]
kategoricke = ["policijska_uprava", "vrsta", "opis"]
X = nezgode[numericke + kategoricke]
y = nezgode["teska"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


def priprema():
    return ColumnTransformer(
        [
            (
                "num",
                make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
                numericke,
            ),
            ("kat", OneHotEncoder(handle_unknown="ignore"), kategoricke),
        ]
    )


print("Trening:", len(X_tr), "nezgoda, test:", len(X_te), "nezgoda\n")

modeli = {
    "logistička regresija": LogisticRegression(max_iter=1000),
    "kNN, k = 5": KNeighborsClassifier(n_neighbors=5),
    "kNN, k = 51": KNeighborsClassifier(n_neighbors=51),
}
for naziv, klasifikator in modeli.items():
    model = Pipeline([("priprema", priprema()), ("model", klasifikator)])
    model.fit(X_tr, y_tr)
    print(f"{naziv:22s} tačnost = {model.score(X_te, y_te):.3f}")

rastojanja = len(X_te) * len(X_tr)
print(f"\nkNN za procenu testa računa {rastojanja:,} rastojanja".replace(",", "."))

## Tumačenje modela

### Zadatak 8. Koeficijenti i odnos šansi

Ispisati koeficijente logističke regresije za vrste nezgoda i njihove eksponente, pa izračunati koliko su puta veće šanse da nezgoda sa pešakom bude teška nego nezgoda sa jednim vozilom.

Uz ista ostala obeležja, šanse da nezgoda bude teška oko 182 puta su veće kod nezgode sa pešakom nego kod nezgode sa jednim vozilom. To je šansa, odnos \(p/(1-p)\), a ne verovatnoća. Kada je verovatnoća blizu 1, šansa raste veoma brzo.

In [ ]:
logisticka_nez = Pipeline(
    [("priprema", priprema()), ("model", LogisticRegression(max_iter=1000))]
)
logisticka_nez.fit(X_tr, y_tr)
nazivi = logisticka_nez.named_steps["priprema"].get_feature_names_out()
koeficijenti = pd.Series(logisticka_nez.named_steps["model"].coef_[0], index=nazivi)

vrste = koeficijenti[koeficijenti.index.str.startswith("kat__vrsta")]
tabela = pd.DataFrame({"theta": vrste.round(2), "e^theta": np.exp(vrste).round(2)})
tabela.index = tabela.index.str.replace("kat__vrsta_", "")
print(tabela.sort_values("theta"))

razlika = vrste["kat__vrsta_SN SA PEŠACIMA"] - vrste["kat__vrsta_SN SA JEDNIM VOZILOM"]
odnos = np.exp(razlika)
print(f"\nOdnos šansi, pešaci naspram jednog vozila: e^{razlika:.2f} = {odnos:.0f}")

### Zadatak 9. Verovatnoća i procena

Za prvih pet test nezgoda ispisati verovatnoću teške nezgode, procenu modela i stvarnu klasu.

Prag 0,5 nije zakon, nego podrazumevana vrednost. Kada je propuštena teška nezgoda skuplja od lažne uzbune, prag se spušta. Kako se prag bira i obrazlaže, tema je vežbe 6.

In [ ]:
verovatnoce = logisticka_nez.predict_proba(X_te.head(5))[:, 1]
procene = logisticka_nez.predict(X_te.head(5))
print(
    pd.DataFrame(
        {
            "vrsta": X_te.head(5)["vrsta"].str.replace("SN SA ", "").values,
            "P(teška)": verovatnoce.round(3),
            "procena": procene,
            "stvarno": y_te.head(5).values,
        }
    ).to_string(index=False)
)

## Zadaci za samostalan rad

### Zadatak 10. Koja je ovo funkcija

Ručno, a zatim kodom, odrediti koju logičku funkciju računa neuron sa parametrima θ = (10, −20, −20).

### Zadatak 11. kNN klasifikacija ručno

Date su tačke (1, 1), (2, 1) i (1, 2) klase 0 i tačke (4, 3), (5, 4) i (4, 4) klase 1. Za novu tačku (3; 2,2) izračunati euklidska rastojanja i odrediti klasu za k = 1, 3 i 5.

### Zadatak 12. kNN sa i bez skaliranja

Na problemu prekoračenja PM10 istrenirati kNN sa k = 15 na četiri meteorološke veličine, jednom bez skaliranja, a jednom sa njim. Uporediti tačnost i standardne devijacije obeležja.

### Zadatak 13. Jednačina granice

Iz parametara logističke regresije iz zadatka 5 izračunati temperaturu na kojoj je, uz vetar od 10 km/h, verovatnoća prekoračenja tačno 0,5. Voditi računa da je model učen nad standardizovanim obeležjima. Rezultat proveriti funkcijom `predict_proba`.

### Zadatak 14. Težinski kNN

Na skupu nezgoda uporediti kNN sa k = 51 kada svi susedi glasaju jednako i kada bliži susedi imaju veći glas (`weights="distance"`).